# ARC-AGI-3 VLM Input Probe

VLM에게 게임 이미지를 줬을 때 무엇을 추론할 수 있는지 보기 전에, 현재 런타임에서 실제로 쓸 수 있는 입력값을 확인하는 노트북입니다.

확인 대상: raw `FrameData`, `latest_frame.frame`, grid 이미지, 상태/레벨/액션 메타데이터, 내부 `Frame` 변환 결과.

In [ ]:
from pathlib import Path
import os
import sys
import json
from pprint import pprint

IN_KAGGLE = Path('/kaggle/input').exists()
COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3')
DEFAULT_ENV_DIRS = [
    COMP_ROOT / 'environment_files',
    Path.cwd() / 'environment_files',
    Path('/kaggle/working/agi-arc/environment_files'),
]
ENV_DIR = next((p for p in DEFAULT_ENV_DIRS if p.exists()), DEFAULT_ENV_DIRS[0])
REPO_ROOT = Path(os.getenv('AGI_ARC_REPO', Path.cwd()))

if IN_KAGGLE and COMP_ROOT.exists():
    wheel_dir = COMP_ROOT / 'arc_agi_3_wheels'
    if wheel_dir.exists():
        !pip install --no-index --find-links {wheel_dir} arc-agi python-dotenv -q

for candidate in [REPO_ROOT / 'src', Path.cwd() / 'src', Path('/kaggle/working/agi-arc/src')]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

print('IN_KAGGLE =', IN_KAGGLE)
print('ENV_DIR =', ENV_DIR)
print('REPO_ROOT =', REPO_ROOT)
print('src on path =', [p for p in sys.path[:5] if 'agi-arc' in p or p.endswith('/src')])

In [ ]:
import numpy as np
try:
    from PIL import Image
except Exception as exc:
    Image = None
    print('PIL unavailable, will use matplotlib fallback:', repr(exc))
from IPython.display import display

ARC_PALETTE = {
    0: (0, 0, 0),
    1: (0, 116, 217),
    2: (255, 65, 54),
    3: (46, 204, 64),
    4: (255, 220, 0),
    5: (170, 80, 220),
    6: (255, 133, 27),
    7: (127, 219, 255),
    8: (135, 135, 135),
    9: (255, 255, 255),
    10: (80, 80, 80),
    11: (180, 180, 180),
    12: (128, 0, 0),
    13: (0, 128, 128),
    14: (128, 128, 0),
    15: (255, 105, 180),
}

def object_attrs(obj):
    names = []
    for name in dir(obj):
        if name.startswith('_'):
            continue
        try:
            value = getattr(obj, name)
        except Exception:
            continue
        if callable(value):
            continue
        names.append(name)
    return names

def as_list_grid(grid):
    if grid is None:
        return []
    if hasattr(grid, 'tolist'):
        grid = grid.tolist()
    return [[int(cell) for cell in row] for row in grid]

def raw_frame_stack(raw_frame):
    return list(getattr(raw_frame, 'frame', []) or [])

def latest_grid_from_raw(raw_frame):
    frames = raw_frame_stack(raw_frame)
    return as_list_grid(frames[-1]) if frames else []

def summarize_grid(grid):
    grid = as_list_grid(grid)
    if not grid:
        return {'shape': [0, 0], 'colors': [], 'nonzero': 0}
    arr = np.asarray(grid, dtype=int)
    return {
        'shape': list(arr.shape),
        'colors': sorted(int(x) for x in np.unique(arr)),
        'nonzero': int(np.count_nonzero(arr)),
        'min': int(arr.min()),
        'max': int(arr.max()),
    }

def grid_to_rgb_array(grid, draw_grid=True):
    grid = as_list_grid(grid)
    if not grid:
        return np.zeros((1, 1, 3), dtype=np.uint8)
    arr = np.asarray(grid, dtype=int)
    rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.uint8)
    for value, color in ARC_PALETTE.items():
        rgb[arr == value] = color
    if draw_grid:
        # Keep this as raw cell RGB. Display scaling is handled separately.
        pass
    return rgb


def grid_to_image(grid, scale=14, draw_grid=True):
    rgb = grid_to_rgb_array(grid, draw_grid=False)
    if Image is None:
        return rgb
    h, w = rgb.shape[:2]
    img = Image.fromarray(rgb, mode='RGB').resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
    if draw_grid and scale >= 8:
        line = (35, 35, 35)
        px = img.load()
        max_x = w * scale - 1
        max_y = h * scale - 1
        for gx in range(0, w * scale, scale):
            for yy in range(h * scale):
                px[min(gx, max_x), yy] = line
        for gy in range(0, h * scale, scale):
            for xx in range(w * scale):
                px[xx, min(gy, max_y)] = line
    return img

def summarize_raw_frame(raw_frame, show_image=True):
    frames = raw_frame_stack(raw_frame)
    grid = latest_grid_from_raw(raw_frame)
    summary = {
        'type': type(raw_frame).__name__,
        'attrs': object_attrs(raw_frame),
        'state': getattr(getattr(raw_frame, 'state', None), 'name', str(getattr(raw_frame, 'state', None))),
        'guid': getattr(raw_frame, 'guid', None),
        'levels_completed': getattr(raw_frame, 'levels_completed', None),
        'win_levels': getattr(raw_frame, 'win_levels', None),
        'available_actions': [getattr(a, 'name', str(a)) for a in list(getattr(raw_frame, 'available_actions', []) or [])],
        'frame_count': len(frames),
        'frame_shapes': [list(np.asarray(as_list_grid(f)).shape) for f in frames[:5]],
        'latest_grid': summarize_grid(grid),
    }
    pprint(summary)
    if show_image:
        display(grid_to_image(grid, scale=12, draw_grid=True))
    return summary

In [ ]:
def collect_one_raw_frame(game_id='bp35', mode='offline', render_mode=None):
    """Arcade wrapper에서 raw FrameData를 직접 받아서 VLM 입력 후보를 확인합니다."""
    from arc_agi import Arcade, OperationMode

    arcade = Arcade(
        arc_api_key=os.getenv('ARC_API_KEY', 'test-key-123'),
        arc_base_url=os.getenv('ARC_BASE_URL', 'http://gateway:8001' if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else 'https://three.arcprize.org'),
        operation_mode=OperationMode(mode.lower()),
        environments_dir=str(ENV_DIR),
        recordings_dir=str(Path('/kaggle/working/recordings') if IN_KAGGLE else Path('recordings')),
    )
    scorecard_id = arcade.create_scorecard(source_url='vlm-input-probe', tags=['vlm-input-probe'])
    try:
        wrapper = arcade.make(
            game_id,
            scorecard_id=scorecard_id,
            save_recording=False,
            include_frame_data=True,
            render_mode=render_mode,
        )
        raw = wrapper.reset()
        print('raw action_space =', [getattr(a, 'name', str(a)) for a in getattr(wrapper, 'action_space', [])])
        summary = summarize_raw_frame(raw, show_image=True)
        return raw, summary
    finally:
        try:
            arcade.close_scorecard(scorecard_id)
        except Exception as exc:
            print('close_scorecard failed:', repr(exc))

# 예시: Kaggle/offline 환경에서 실행
# raw, summary = collect_one_raw_frame(game_id='bp35', mode='offline')
# raw, summary = collect_one_raw_frame(game_id='cd82', mode='offline')

In [ ]:
def inspect_latest_frame_from_agent(latest_frame):
    """Agent.choose_action(frames, latest_frame) 안에서 latest_frame을 넘겨 호출하는 용도입니다."""
    return summarize_raw_frame(latest_frame, show_image=True)

def explain_available_vlm_inputs():
    rows = [
        ('latest_frame.frame', 'list/array of rendered grids. 보통 마지막 원소가 현재 화면이며 VLM 이미지로 변환 가능'),
        ('all frames in latest_frame.frame', '한 action 안에서 여러 렌더 프레임이 오면 짧은 애니메이션/변화 단서로 사용 가능'),
        ('latest_frame.state', 'NOT_PLAYED / NOT_FINISHED / WIN / GAME_OVER 같은 게임 상태'),
        ('latest_frame.available_actions', '현재 게임에서 허용되는 ACTION1~ACTION7/RESET 목록'),
        ('latest_frame.levels_completed', '완료한 레벨 수. reward/progress 판단에 사용 가능'),
        ('latest_frame.win_levels', '전체 또는 승리 관련 레벨 메타데이터로 보이는 값'),
        ('latest_frame.guid', '현재 episode/environment instance 식별자'),
        ('grid statistics', 'shape, colors, object count, nonzero cells 등 텍스트 prompt 보조 정보'),
        ('previous frames/actions memory', '직접 저장해야 하는 history. 대회가 자동으로 규칙 설명을 주지는 않음'),
        ('environment_files/*.py', 'public/offline 분석용 소스. private 평가에서도 파일이 제공되는지는 대회 환경에 의존'),
    ]
    for name, desc in rows:
        print(f'- {name}: {desc}')

explain_available_vlm_inputs()